---
## Step 5: (Stretch) Dockerize

```bash
# Create Dockerfile in current directory
# docker build -t churn-api .
# docker run -p 8000:8000 churn-api
# Test it again
```

## What I Built
- FastAPI ML service with Pydantic validation
- /predict, /health, /model/info endpoints
- Load tests showing latency characteristics

## What I Learned
-

In [ ]:
# TODO: Simulate concurrent requests to test performance
# import concurrent.futures, time
#
# def single_request(i):
#     payload = {"tenure_months": i % 72 + 1, "monthly_charges": 50 + i % 100,
#                "total_charges": 1000, "contract_type": "Month-to-month"}
#     start = time.time()
#     r = requests.post(f"{BASE}/predict", json=payload)
#     return time.time() - start
#
# with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
#     times = list(executor.map(single_request, range(100)))
#
# print(f"100 requests: mean={np.mean(times)*1000:.1f}ms, p95={np.percentile(times,95)*1000:.1f}ms")

---
## Step 4: Load Test

In [ ]:
# TODO: Write tests for each endpoint using requests
# import requests
# BASE = "http://localhost:8000"
#
# Test 1: Health check
# Test 2: Single prediction (high risk customer)
# Test 3: Single prediction (low risk customer)
# Test 4: Validation error (invalid contract_type)
# Test 5: Boundary values (tenure=0, monthly=0)

# Track results
tests_passed = 0
tests_total = 5

# TODO: implement tests

---
## Step 3: Test All Endpoints

---
## Step 2: Start the Server

In a terminal:
```bash
uvicorn main:app --reload --port 8000
```
Then open http://localhost:8000/docs

In [ ]:
# TODO: Write main.py with:
# 1. Load model.pkl at startup
# 2. POST /predict - single prediction endpoint
#    Input: tenure_months, monthly_charges, total_charges, contract_type, num_products, support_calls
#    Output: churn_probability, churn_predicted, risk_tier
# 3. GET /health - health check endpoint
# 4. GET /model/info - model metadata endpoint

main_app = '''
from fastapi import FastAPI
import pickle

app = FastAPI(title="Churn API")

# TODO: Load model at startup
# TODO: Define request/response schemas with Pydantic
# TODO: Implement /predict endpoint
# TODO: Implement /health endpoint
# TODO: Implement /model/info endpoint
'''

# Uncomment to create the file
# with open('main.py', 'w') as f:
#     f.write(main_app)
# print("Created main.py")

---
## Step 1: Write the FastAPI App

Create `main.py` in the current directory.

In [ ]:
import pickle, json
import numpy as np, pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder

np.random.seed(42)
n = 5000
tenure = np.random.exponential(24, n).clip(1, 72).astype(int)
monthly = np.random.normal(65, 25, n).clip(20, 150)
contracts = np.random.choice([0, 1, 2], n, p=[0.55, 0.25, 0.20])
num_prod = np.random.randint(1, 6, n)
calls = np.random.poisson(2, n)
total = tenure * monthly

prob = 1 / (1 + np.exp(-(-2 + 0.05*(monthly-65) - 0.03*tenure + 0.1*calls)))
y = (np.random.random(n) < prob).astype(int)

X = pd.DataFrame({'tenure_months': tenure, 'monthly_charges': monthly,
                  'total_charges': total, 'contract_encoded': contracts,
                  'num_products': num_prod, 'support_calls': calls})

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
model.fit(X_train, y_train)
auc = roc_auc_score(y_val, model.predict_proba(X_val)[:,1])
print(f"Model AUC: {auc:.4f}")

with open('model.pkl', 'wb') as f:
    pickle.dump({'model': model, 'feature_cols': X.columns.tolist(), 'auc': auc}, f)
print("Saved: model.pkl")

# Mini-Project 2: FastAPI ML Service

## Goal
Build and test a complete FastAPI ML service:
1. Train a simple churn model
2. Save it with pickle
3. Write a FastAPI app to serve it
4. Write tests for all endpoints
5. (Stretch) Containerize with Docker

## Estimated Time: 2.5 hours